# RAG PDF Question-Answering Chatbot

This notebook demonstrates how a PDF document can be converted into a retrieval-based question-answering system. It loads the paper, splits it into searchable chunks, creates embeddings, stores them in a vector database, and uses the retrieved context to answer questions with a language model.

The code is organized into a clear pipeline so that each stage—document loading, chunking, vector search, prompting, and generation—can be understood independently.

In [1]:
# Import the libraries required for loading, splitting, indexing, and answering PDF questions.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

# Load environment variables used for API authentication.
load_dotenv()

C:\Users\vishe\AppData\Local\Temp\ipykernel_16420\791165039.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
# Load the PDF and verify that the document was extracted into readable text pages.
pdf_loader = PyPDFLoader("../data/research_paper.pdf")
loaded_pdf_documents = pdf_loader.load()
print("Number of documents loaded:", len(loaded_pdf_documents))

Number of documents loaded: 67


In [3]:
# Split the PDF into overlapping chunks so meaningful context can be retrieved.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunked_pdf_documents = text_splitter.split_documents(loaded_pdf_documents)
print("Number of chunks created:", len(chunked_pdf_documents))

Number of chunks created: 149


In [4]:
# Create embeddings and save the vector index for fast semantic similarity search.
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(
    documents=chunked_pdf_documents,
    embedding=embeddings
)
vector_store.save_local("./vector_store_pdf_chatbot")

In [5]:
# Retrieve the three most relevant chunks for the user's question.
def get_relevant_context(user_query: str) -> dict:
    matching_documents = vector_store.similarity_search(user_query, k=3)
    retrieved_context = ""
    for match in matching_documents:
        retrieved_context += match.page_content

    return {
        "query": user_query,
        "context": retrieved_context
    }

In [6]:
# Define the answer prompt so the model responds only from the retrieved context.
from langchain_core.prompts import PromptTemplate

answer_prompt = PromptTemplate.from_template("""
    Answer the question using only the context.
    If you do not know the answer, say "I don't know".

    Question: {query}
    Context: {context}
""")

In [7]:
# Initialize the language model used to generate the final answer.
llm = ChatOpenAI(model="gpt-6-luna")

In [8]:
# Combine retrieval and prompt generation into a single LangChain pipeline.
rag_chain = get_relevant_context | answer_prompt | llm

In [9]:
# Run the complete RAG pipeline with an example question.
example_question = "What is attention?"
response = rag_chain.invoke(example_question)
print(response.content)

Attention is a function that maps a query and a set of key-value pairs to an output. In a Transformer, each token is converted into a query, key, and value. To compute a token’s representation, the model uses the query to score the keys, then takes a weighted sum of the corresponding values. The scores are based on a scaled dot product.


In [10]:
example_question = "How does this paper relate to Jev models"
response = rag_chain.invoke(example_question)
print(response.content)

I don’t know. The context doesn’t mention JEV models.
